In [ ]:
NGROK_AUTH_TOKEN=''

In [2]:
!mkdir audio_chunk
!mkdir audio_chunk/filemp3_chunk
!mkdir audio/
!mkdir text

mkdir: cannot create directory ‘audio/’: File exists
mkdir: cannot create directory ‘text’: File exists


In [42]:
!pip install yt-dlp pytube

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.5 MB/s eta 0:00:00


In [8]:
!pip install bertopic underthesea umap-learn hdbscan pyvi

  Using cached bertopic-0.17.4-py3-none-any.whl.metadata (24 kB)
  Using cached underthesea-9.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.3 MB/s eta 0:00:00


In [10]:
!pip install fastapi uvicorn pyngrok nest-asyncio pyvi sentence-transformers bertopic -q

In [26]:
!pip install asyncio

In [113]:
import torch
import librosa
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import subprocess
from pydub import AudioSegment
import os
class Mp4_to_text:
    def __init__(self,model_id="vinai/PhoWhisper-small", language = "vi", time_chunk_mp3=30 * 1000, folder_path_chunk = "/content/audio_chunk/filemp3_chunk", output_file_path_mp3 = "/content/audio/output3.mp3",
                 output_file_path_txt = "/content/text/output.txt", gpu=False):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")

        self.model=AutoModelForSpeechSeq2Seq.from_pretrained(model_id
                                                             ,torch_dtype=torch.float16 if self.device == "cuda" else torch.float32)
        self.model.config.tie_word_embeddings = False
        self.processor = AutoProcessor.from_pretrained(model_id)
        print('load model successfully')
        # ngôn ngữ của video đầu vào hiện tại chỉ lấy tiếng việt. Trong tương lai sẽ update và thêm các model để xử lý video tiếng nước ngoài
        self.language = language
        # đường dẫn của video mp4 đầu vào
        # độ dài mỗi chunk để model trích xuất khuyến nghị không nên sửa để tránh model bị quá tham số -> lỗi
        self.chunk_length_ms = time_chunk_mp3
        # đường dẫn của folder tổng lưu trữ chunk
        self.base_folder = folder_path_chunk
        # đường dẫn của file audio mp3 sau khi được chuyển từ video mp4
        self.op_file_mp3 = output_file_path_mp3
        # đường dẫn kết quả trả về file txt
        self.op_file_txt = output_file_path_txt
        # nếu máy có gpu thì là True không thì là False tối ưu cho tốc độ nếu có gpu
        self.gpu=torch.cuda.is_available()
        # ví dụ:
        # self.language = "vi"
        # self.ip_path = "/kaggle/input/datasets/fasgadhsxnzmjj/mp4-dataset/treemp4.mp4"
        # self.chunk_length_ms = 30 * 1000
        # self.base_folder = "/kaggle/working/filemp3_chunk"
        # self.op_file_mp3 = "/kaggle/working/output2.mp3"
        # self.gpu = "True"

    def split_audio_into_chunks(self, file_path):
        folder = os.path.join(
            self.base_folder,
            os.path.splitext(os.path.basename(file_path))[0]
        )

        os.makedirs(folder, exist_ok=True)

        audio = AudioSegment.from_file(file_path)
        duration = len(audio)

        for i, start in enumerate(range(0, duration, self.chunk_length_ms)):
            chunk = audio[start:start + self.chunk_length_ms]
            chunk_path = os.path.join(folder, f"chunk_{i}.mp3")
            chunk.export(chunk_path, format="mp3")

        return folder

    def mp4_to_mp3(self,ip_path):
        command = [
            "ffmpeg", "-i",ip_path,
            "-vn",
            "-acodec", "libmp3lame",
            "-ab", "192k",
            self.op_file_mp3
        ]
        subprocess.run(command, check=True)

    def mp3_to_text_phoWhisper(self, file_mp3):
        audio, sr = librosa.load(file_mp3, sr=16000)

        inputs = self.processor(
            audio,
            sampling_rate=sr,
            return_tensors="pt"
        )

        with torch.no_grad():
            ids = self.model.generate(inputs["input_features"])

        return self.processor.batch_decode(ids, skip_special_tokens=True)[0]

    def batch_mp3_to_text(self, file_list):
        # dùng nếu máy có GPU
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(device)

        audios = []

        # Load toàn bộ audio
        for file_mp3 in file_list:
            audio, sr = librosa.load(file_mp3, sr=16000)
            audios.append(audio)

        # Convert batch
        inputs = self.processor(
            audios,
            sampling_rate=16000,
            return_tensors="pt",
            padding='max_length',# QUAN TRỌNG
            return_attention_mask=True,
            truncation=True
        )

        # Đưa lên GPU
        inputs = {k: v.to(device) for k, v in inputs.items()}
        inputs['input_features']=inputs['input_features'].half()

        # Inference 1 lần
        with torch.no_grad():
            generated_ids = self.model.generate(
                inputs["input_features"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False
            )

        # Decode batch
        texts = self.processor.batch_decode(generated_ids, skip_special_tokens=True)

        return texts

    def save_to_txt(self, text, output_path="output.txt"):
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)

    def op_vi(self,input_file_path_mp4):
        result = []

        self.mp4_to_mp3(input_file_path_mp4)
        folder = self.split_audio_into_chunks(self.op_file_mp3)

        files = sorted(
            [f for f in os.listdir(folder) if f.endswith(".mp3")],
            key=lambda x: int(x.split("_")[-1].split(".")[0])
        )

        if not self.gpu:
            for f in files:
                path = os.path.join(folder, f)
                text = self.mp3_to_text_phoWhisper(path)
                result.append(text)

        if self.gpu:
            file_paths = [os.path.join(folder, f) for f in files]
            # chia batch (rất quan trọng nếu nhiều file)
            batch_size = 4

            for i in range(0, len(file_paths), batch_size):
                batch_files = file_paths[i:i + batch_size]
                texts = self.batch_mp3_to_text(batch_files)
                result.extend(texts)

        # return result
        full_text = " ".join(result)
        self.save_to_txt(full_text, self.op_file_txt)
        return full_text


# ngôn ngữ của video đầu vào hiện tại chỉ lấy tiếng việt. Trong tương lai sẽ update và thêm các model để xử lý video tiếng nước ngoài

# đường dẫn của video mp4 đầu vào

# độ dài mỗi chunk để model trích xuất khuyến nghị không nên sửa để tránh model bị quá tham số -> lỗi
# đường dẫn của folder tổng lưu trữ chunk

# đường dẫn của file audio mp3 sau khi được chuyển từ video mp4

# đường dẫn kết quả trả về file txt

# nếu máy có gpu thì là True không thì là False tối ưu cho tốc độ nếu có gpu



In [34]:
import re
import os
from typing import List
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 150
MIN_NEW_TOKENS = 60
CHUNK_SIZE = 250
class Summarizer:
    def __init__(self,model_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path,torch_dtype=torch.float16 if self.device == "cuda" else torch.float32).to(self.device)
        print("Load summary model successfully !!!")
        self.model.eval()

    def clean_subtitle_text(self,text: str) -> str:
        lines = text.splitlines()
        cleaned_lines = []

        for line in lines:
            line = line.strip()

            if not line:
                continue
            if re.fullmatch(r"\d+", line):
                continue
            if re.search(r"\d{2}:\d{2}:\d{2}", line):
                continue
            if line.upper() == "WEBVTT":
                continue

            cleaned_lines.append(line)

        text = " ".join(cleaned_lines)
        text = re.sub(r"\[.*?\]", " ", text)
        text = re.sub(r"\(.*?\)", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

        return text

    def read_text_file(self,file_path: str) -> str:
        ext = os.path.splitext(file_path)[1].lower()

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        if ext == ".txt":
            return content

        if ext in [".srt", ".vtt"]:
            return self.clean_subtitle_text(content)

        return content

    def normalize_text(self,text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def split_sentences(self,text: str):
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if s.strip()]

    def split_text_into_chunks(self,text: str, max_tokens: int = CHUNK_SIZE) -> List[str]:
        sentences = self.split_sentences(text)

        chunks = []
        current_chunk = ""

        for sentence in sentences:
            temp = current_chunk + " " + sentence

            if len(temp) < 1000:
                token_count = len(self.tokenizer.encode(temp, add_special_tokens=False))
            else:
                token_count = max_tokens + 1

            if token_count <= max_tokens:
                current_chunk = temp
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence

        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks
    def summarize_chunk(self,text: str) -> str:
        text = self.normalize_text(text)

        inputs = self.tokenizer(
            text,
            max_length=MAX_INPUT_TOKENS,
            truncation=True,
            return_tensors="pt"
        )
        if torch.cuda.is_available():
          inputs={k:v.to(self.device) for k,v in inputs.items()}
        with torch.no_grad():
            summary_ids = self.model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=MAX_NEW_TOKENS,
                min_new_tokens=MIN_NEW_TOKENS,
                num_beams=3,
                length_penalty=1.0,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        return summary.strip()

    # 9. TÓM TẮT VĂN BẢN DÀI
    def summarize_long_text(self,text: str) -> str:
        text = self.normalize_text(text)

        token_len = len(self.tokenizer.encode(text, add_special_tokens=False))

        # nếu ngắn → tóm tắt luôn
        if token_len <= MAX_INPUT_TOKENS:
            return self.summarize_chunk(text)

        # chia đoạn
        chunks = self.split_text_into_chunks(text)
        print(f"Số đoạn: {len(chunks)}")

        chunk_summaries = []

        for i, chunk in enumerate(chunks, 1):
            print(f"Đang xử lý đoạn {i}/{len(chunks)}...")
            try:
                summary = self.summarize_chunk(chunk)
                if summary:
                    chunk_summaries.append(summary)
            except Exception as e:
                print(f"Lỗi đoạn {i}: {e}")

        #  GHÉP LẠI
        final_summary = " ".join(chunk_summaries)

        # clean format
        final_summary = final_summary.replace(" .", ".")
        final_summary = final_summary.replace(" ,", ",")

        return final_summary.strip()
    def run(self,text):
        summary = self.summarize_long_text(text)
        return summary

In [60]:
vn_stopwords=['a lô', 'a ha', 'ai', 'ai ai', 'ai nấy', 'ai đó', 'alô', 'amen', 'anh', 'anh ấy', 'ba', 'ba ba', 'ba bản', 'ba cùng', 'ba họ', 'ba ngày', 'ba ngôi', 'ba tăng', 'bao giờ', 'bao lâu', 'bao nhiêu', 'bao nả', 'bay biến', 'biết', 'biết bao', 'biết bao nhiêu', 'biết chắc', 'biết chừng nào', 'biết mình', 'biết mấy', 'biết thế', 'biết trước', 'biết việc', 'biết đâu', 'biết đâu chừng', 'biết đâu đấy', 'biết được', 'buổi', 'buổi làm', 'buổi mới', 'buổi ngày', 'buổi sớm', 'bà', 'bà ấy', 'bài', 'bài bác', 'bài bỏ', 'bài cái', 'bác', 'bán', 'bán cấp', 'bán dạ', 'bán thế', 'bây bẩy', 'bây chừ', 'bây giờ', 'bây nhiêu', 'bèn', 'béng', 'bên', 'bên bị', 'bên có', 'bên cạnh', 'bông', 'bước', 'bước khỏi', 'bước tới', 'bước đi', 'bạn', 'bản', 'bản bộ', 'bản riêng', 'bản thân', 'bản ý', 'bất chợt', 'bất cứ', 'bất giác', 'bất kì', 'bất kể', 'bất kỳ', 'bất luận', 'bất ngờ', 'bất nhược', 'bất quá', 'bất quá chỉ', 'bất thình lình', 'bất tử', 'bất đồ', 'bấy', 'bấy chầy', 'bấy chừ', 'bấy giờ', 'bấy lâu', 'bấy lâu nay', 'bấy nay', 'bấy nhiêu', 'bập bà bập bõm', 'bập bõm', 'bắt đầu', 'bắt đầu từ', 'bằng', 'bằng cứ', 'bằng không', 'bằng người', 'bằng nhau', 'bằng như', 'bằng nào', 'bằng nấy', 'bằng vào', 'bằng được', 'bằng ấy', 'bển', 'bệt', 'bị', 'bị chú', 'bị vì', 'bỏ', 'bỏ bà', 'bỏ cha', 'bỏ cuộc', 'bỏ không', 'bỏ lại', 'bỏ mình', 'bỏ mất', 'bỏ mẹ', 'bỏ nhỏ', 'bỏ quá', 'bỏ ra', 'bỏ riêng', 'bỏ việc', 'bỏ xa', 'bỗng', 'bỗng chốc', 'bỗng dưng', 'bỗng không', 'bỗng nhiên', 'bỗng nhưng', 'bỗng thấy', 'bỗng đâu', 'bộ', 'bộ thuộc', 'bộ điều', 'bội phần', 'bớ', 'bởi', 'bởi ai', 'bởi chưng', 'bởi nhưng', 'bởi sao', 'bởi thế', 'bởi thế cho nên', 'bởi tại', 'bởi vì', 'bởi vậy', 'bởi đâu', 'bức', 'cao', 'cao lâu', 'cao ráo', 'cao răng', 'cao sang', 'cao số', 'cao thấp', 'cao thế', 'cao xa', 'cha', 'cha chả', 'chao ôi', 'chia sẻ', 'chiếc', 'cho', 'cho biết', 'cho chắc', 'cho hay', 'cho nhau', 'cho nên', 'cho rằng', 'cho rồi', 'cho thấy', 'cho tin', 'cho tới', 'cho tới khi', 'cho về', 'cho ăn', 'cho đang', 'cho được', 'cho đến', 'cho đến khi', 'cho đến nỗi', 'choa', 'chu cha', 'chui cha', 'chung', 'chung cho', 'chung chung', 'chung cuộc', 'chung cục', 'chung nhau', 'chung qui', 'chung quy', 'chung quy lại', 'chung ái', 'chuyển', 'chuyển tự', 'chuyển đạt', 'chuyện', 'chuẩn bị', 'chành chạnh', 'chí chết', 'chính', 'chính bản', 'chính giữa', 'chính là', 'chính thị', 'chính điểm', 'chùn chùn', 'chùn chũn', 'chú', 'chú dẫn', 'chú khách', 'chú mày', 'chú mình', 'chúng', 'chúng mình', 'chúng ta', 'chúng tôi', 'chúng ông', 'chăn chắn', 'chăng', 'chăng chắc', 'chăng nữa', 'chơi', 'chơi họ', 'chưa', 'chưa bao giờ', 'chưa chắc', 'chưa có', 'chưa cần', 'chưa dùng', 'chưa dễ', 'chưa kể', 'chưa tính', 'chưa từng', 'chầm chập', 'chậc', 'chắc', 'chắc chắn', 'chắc dạ', 'chắc hẳn', 'chắc lòng', 'chắc người', 'chắc vào', 'chắc ăn', 'chẳng lẽ', 'chẳng những', 'chẳng nữa', 'chẳng phải', 'chết nỗi', 'chết thật', 'chết tiệt', 'chỉ', 'chỉ chính', 'chỉ có', 'chỉ là', 'chỉ tên', 'chỉn', 'chị', 'chị bộ', 'chị ấy', 'chịu', 'chịu chưa', 'chịu lời', 'chịu tốt', 'chịu ăn', 'chọn', 'chọn bên', 'chọn ra', 'chốc chốc', 'chớ', 'chớ chi', 'chớ gì', 'chớ không', 'chớ kể', 'chớ như', 'chợt', 'chợt nghe', 'chợt nhìn', 'chủn', 'chứ', 'chứ ai', 'chứ còn', 'chứ gì', 'chứ không', 'chứ không phải', 'chứ lại', 'chứ lị', 'chứ như', 'chứ sao', 'coi bộ', 'coi mòi', 'con', 'con con', 'con dạ', 'con nhà', 'con tính', 'cu cậu', 'cuối', 'cuối cùng', 'cuối điểm', 'cuốn', 'cuộc', 'càng', 'càng càng', 'càng hay', 'cá nhân', 'các', 'các cậu', 'cách', 'cách bức', 'cách không', 'cách nhau', 'cách đều', 'cái', 'cái gì', 'cái họ', 'cái đã', 'cái đó', 'cái ấy', 'câu hỏi', 'cây', 'cây nước', 'còn', 'còn như', 'còn nữa', 'còn thời gian', 'còn về', 'có', 'có ai', 'có chuyện', 'có chăng', 'có chăng là', 'có chứ', 'có cơ', 'có dễ', 'có họ', 'có khi', 'có ngày', 'có người', 'có nhiều', 'có nhà', 'có phải', 'có số', 'có tháng', 'có thế', 'có thể', 'có vẻ', 'có ý', 'có ăn', 'có điều', 'có điều kiện', 'có đáng', 'có đâu', 'có được', 'cóc khô', 'cô', 'cô mình', 'cô quả', 'cô tăng', 'cô ấy', 'công nhiên', 'cùng', 'cùng chung', 'cùng cực', 'cùng nhau', 'cùng tuổi', 'cùng tột', 'cùng với', 'cùng ăn', 'căn', 'căn cái', 'căn cắt', 'căn tính', 'cũng', 'cũng như', 'cũng nên', 'cũng thế', 'cũng vậy', 'cũng vậy thôi', 'cũng được', 'cơ', 'cơ chỉ', 'cơ chừng', 'cơ cùng', 'cơ dẫn', 'cơ hồ', 'cơ hội', 'cơ mà', 'cơn', 'cả', 'cả nghe', 'cả nghĩ', 'cả ngày', 'cả người', 'cả nhà', 'cả năm', 'cả thảy', 'cả thể', 'cả tin', 'cả ăn', 'cả đến', 'cảm thấy', 'cảm ơn', 'cấp', 'cấp số', 'cấp trực tiếp', 'cần', 'cần cấp', 'cần gì', 'cần số', 'cật lực', 'cật sức', 'cậu', 'cổ lai', 'cụ thể', 'cụ thể là', 'cụ thể như', 'của', 'của ngọt', 'của tin', 'cứ', 'cứ như', 'cứ việc', 'cứ điểm', 'cực lực', 'do', 'do vì', 'do vậy', 'do đó', 'duy', 'duy chỉ', 'duy có', 'dài', 'dài lời', 'dài ra', 'dành', 'dành dành', 'dào', 'dì', 'dù', 'dù cho', 'dù dì', 'dù gì', 'dù rằng', 'dù sao', 'dùng', 'dùng cho', 'dùng hết', 'dùng làm', 'dùng đến', 'dưới', 'dưới nước', 'dạ', 'dạ bán', 'dạ con', 'dạ dài', 'dạ dạ', 'dạ khách', 'dần dà', 'dần dần', 'dầu sao', 'dẫn', 'dẫu', 'dẫu mà', 'dẫu rằng', 'dẫu sao', 'dễ', 'dễ dùng', 'dễ gì', 'dễ khiến', 'dễ nghe', 'dễ ngươi', 'dễ như chơi', 'dễ sợ', 'dễ sử dụng', 'dễ thường', 'dễ thấy', 'dễ ăn', 'dễ đâu', 'dở chừng', 'dữ', 'dữ cách', 'em', 'em em', 'giá trị', 'giá trị thực tế', 'giảm', 'giảm chính', 'giảm thấp', 'giảm thế', 'giống', 'giống người', 'giống nhau', 'giống như', 'giờ', 'giờ lâu', 'giờ này', 'giờ đi', 'giờ đây', 'giờ đến', 'giữ', 'giữ lấy', 'giữ ý', 'giữa', 'giữa lúc', 'gây', 'gây cho', 'gây giống', 'gây ra', 'gây thêm', 'gì', 'gì gì', 'gì đó', 'gần', 'gần bên', 'gần hết', 'gần ngày', 'gần như', 'gần xa', 'gần đây', 'gần đến', 'gặp', 'gặp khó khăn', 'gặp phải', 'gồm', 'hay', 'hay biết', 'hay hay', 'hay không', 'hay là', 'hay làm', 'hay nhỉ', 'hay nói', 'hay sao', 'hay tin', 'hay đâu', 'hiểu', 'hiện nay', 'hiện tại', 'hoàn toàn', 'hoặc', 'hoặc là', 'hãy', 'hãy còn', 'hơn', 'hơn cả', 'hơn hết', 'hơn là', 'hơn nữa', 'hơn trước', 'hầu hết', 'hết', 'hết chuyện', 'hết cả', 'hết của', 'hết nói', 'hết ráo', 'hết rồi', 'hết ý', 'họ', 'họ gần', 'họ xa', 'hỏi', 'hỏi lại', 'hỏi xem', 'hỏi xin', 'hỗ trợ', 'khi', 'khi khác', 'khi không', 'khi nào', 'khi nên', 'khi trước', 'khiến', 'khoảng', 'khoảng cách', 'khoảng không', 'khá', 'khá tốt', 'khác', 'khác gì', 'khác khác', 'khác nhau', 'khác nào', 'khác thường', 'khác xa', 'khách', 'khó', 'khó biết', 'khó chơi', 'khó khăn', 'khó làm', 'khó mở', 'khó nghe', 'khó nghĩ', 'khó nói', 'khó thấy', 'khó tránh', 'không', 'không ai', 'không bao giờ', 'không bao lâu', 'không biết', 'không bán', 'không chỉ', 'không còn', 'không có', 'không có gì', 'không cùng', 'không cần', 'không cứ', 'không dùng', 'không gì', 'không hay', 'không khỏi', 'không kể', 'không ngoài', 'không nhận', 'không những', 'không phải', 'không phải không', 'không thể', 'không tính', 'không điều kiện', 'không được', 'không đầy', 'không để', 'khẳng định', 'khỏi', 'khỏi nói', 'kể', 'kể cả', 'kể như', 'kể tới', 'kể từ', 'liên quan', 'loại', 'loại từ', 'luôn', 'luôn cả', 'luôn luôn', 'luôn tay', 'là', 'là cùng', 'là là', 'là nhiều', 'là phải', 'là thế nào', 'là vì', 'là ít', 'làm', 'làm bằng', 'làm cho', 'làm dần dần', 'làm gì', 'làm lòng', 'làm lại', 'làm lấy', 'làm mất', 'làm ngay', 'làm như', 'làm nên', 'làm ra', 'làm riêng', 'làm sao', 'làm theo', 'làm thế nào', 'làm tin', 'làm tôi', 'làm tăng', 'làm tại', 'làm tắp lự', 'làm vì', 'làm đúng', 'làm được', 'lâu', 'lâu các', 'lâu lâu', 'lâu nay', 'lâu ngày', 'lên', 'lên cao', 'lên cơn', 'lên mạnh', 'lên ngôi', 'lên nước', 'lên số', 'lên xuống', 'lên đến', 'lòng', 'lòng không', 'lúc', 'lúc khác', 'lúc lâu', 'lúc nào', 'lúc này', 'lúc sáng', 'lúc trước', 'lúc đi', 'lúc đó', 'lúc đến', 'lúc ấy', 'lý do', 'lượng', 'lượng cả', 'lượng số', 'lượng từ', 'lại', 'lại bộ', 'lại cái', 'lại còn', 'lại giống', 'lại làm', 'lại người', 'lại nói', 'lại nữa', 'lại quả', 'lại thôi', 'lại ăn', 'lại đây', 'lấy', 'lấy có', 'lấy cả', 'lấy giống', 'lấy làm', 'lấy lý do', 'lấy lại', 'lấy ra', 'lấy ráo', 'lấy sau', 'lấy số', 'lấy thêm', 'lấy thế', 'lấy vào', 'lấy xuống', 'lấy được', 'lấy để', 'lần', 'lần khác', 'lần lần', 'lần nào', 'lần này', 'lần sang', 'lần sau', 'lần theo', 'lần trước', 'lần tìm', 'lớn', 'lớn lên', 'lớn nhỏ', 'lời', 'lời chú', 'lời nói', 'mang', 'mang lại', 'mang mang', 'mang nặng', 'mang về', 'muốn', 'mà', 'mà cả', 'mà không', 'mà lại', 'mà thôi', 'mà vẫn', 'mình', 'mạnh', 'mất', 'mất còn', 'mọi', 'mọi giờ', 'mọi khi', 'mọi lúc', 'mọi người', 'mọi nơi', 'mọi sự', 'mọi thứ', 'mọi việc', 'mối', 'mỗi', 'mỗi lúc', 'mỗi lần', 'mỗi một', 'mỗi ngày', 'mỗi người', 'một', 'một cách', 'một cơn', 'một khi', 'một lúc', 'một số', 'một vài', 'một ít', 'mới', 'mới hay', 'mới rồi', 'mới đây', 'mở', 'mở mang', 'mở nước', 'mở ra', 'mợ', 'mức', 'nay', 'ngay', 'ngay bây giờ', 'ngay cả', 'ngay khi', 'ngay khi đến', 'ngay lúc', 'ngay lúc này', 'ngay lập tức', 'ngay thật', 'ngay tức khắc', 'ngay tức thì', 'ngay từ', 'nghe', 'nghe chừng', 'nghe hiểu', 'nghe không', 'nghe lại', 'nghe nhìn', 'nghe như', 'nghe nói', 'nghe ra', 'nghe rõ', 'nghe thấy', 'nghe tin', 'nghe trực tiếp', 'nghe đâu', 'nghe đâu như', 'nghe được', 'nghen', 'nghiễm nhiên', 'nghĩ', 'nghĩ lại', 'nghĩ ra', 'nghĩ tới', 'nghĩ xa', 'nghĩ đến', 'nghỉm', 'ngoài', 'ngoài này', 'ngoài ra', 'ngoài xa', 'ngoải', 'nguồn', 'ngày', 'ngày càng', 'ngày cấp', 'ngày giờ', 'ngày ngày', 'ngày nào', 'ngày này', 'ngày nọ', 'ngày qua', 'ngày rày', 'ngày tháng', 'ngày xưa', 'ngày xửa', 'ngày đến', 'ngày ấy', 'ngôi', 'ngôi nhà', 'ngôi thứ', 'ngõ hầu', 'ngăn ngắt', 'ngươi', 'người', 'người hỏi', 'người khác', 'người khách', 'người mình', 'người nghe', 'người người', 'người nhận', 'ngọn', 'ngọn nguồn', 'ngọt', 'ngồi', 'ngồi bệt', 'ngồi không', 'ngồi sau', 'ngồi trệt', 'ngộ nhỡ', 'nhanh', 'nhanh lên', 'nhanh tay', 'nhau', 'nhiên hậu', 'nhiều', 'nhiều ít', 'nhiệt liệt', 'nhung nhăng', 'nhà', 'nhà chung', 'nhà khó', 'nhà làm', 'nhà ngoài', 'nhà ngươi', 'nhà tôi', 'nhà việc', 'nhân dịp', 'nhân tiện', 'nhé', 'nhìn', 'nhìn chung', 'nhìn lại', 'nhìn nhận', 'nhìn theo', 'nhìn thấy', 'nhìn xuống', 'nhóm', 'nhón nhén', 'như', 'như ai', 'như chơi', 'như không', 'như là', 'như nhau', 'như quả', 'như sau', 'như thường', 'như thế', 'như thế nào', 'như thể', 'như trên', 'như trước', 'như tuồng', 'như vậy', 'như ý', 'nhưng', 'nhưng mà', 'nhược bằng', 'nhất', 'nhất loạt', 'nhất luật', 'nhất là', 'nhất mực', 'nhất nhất', 'nhất quyết', 'nhất sinh', 'nhất thiết', 'nhất thì', 'nhất tâm', 'nhất tề', 'nhất đán', 'nhất định', 'nhận', 'nhận biết', 'nhận họ', 'nhận làm', 'nhận nhau', 'nhận ra', 'nhận thấy', 'nhận việc', 'nhận được', 'nhằm', 'nhằm khi', 'nhằm lúc', 'nhằm vào', 'nhằm để', 'nhỉ', 'nhỏ', 'nhỏ người', 'nhớ', 'nhớ bập bõm', 'nhớ lại', 'nhớ lấy', 'nhớ ra', 'nhờ', 'nhờ chuyển', 'nhờ có', 'nhờ nhờ', 'nhờ đó', 'nhỡ ra', 'những', 'những ai', 'những khi', 'những là', 'những lúc', 'những muốn', 'những như', 'nào', 'nào cũng', 'nào hay', 'nào là', 'nào phải', 'nào đâu', 'nào đó', 'này', 'này nọ', 'nên', 'nên chi', 'nên chăng', 'nên làm', 'nên người', 'nên tránh', 'nó', 'nóc', 'nói', 'nói bông', 'nói chung', 'nói khó', 'nói là', 'nói lên', 'nói lại', 'nói nhỏ', 'nói phải', 'nói qua', 'nói ra', 'nói riêng', 'nói rõ', 'nói thêm', 'nói thật', 'nói toẹt', 'nói trước', 'nói tốt', 'nói với', 'nói xa', 'nói ý', 'nói đến', 'nói đủ', 'năm', 'năm tháng', 'nơi', 'nơi nơi', 'nước', 'nước bài', 'nước cùng', 'nước lên', 'nước nặng', 'nước quả', 'nước xuống', 'nước ăn', 'nước đến', 'nấy', 'nặng', 'nặng căn', 'nặng mình', 'nặng về', 'nếu', 'nếu có', 'nếu cần', 'nếu không', 'nếu mà', 'nếu như', 'nếu thế', 'nếu vậy', 'nếu được', 'nền', 'nọ', 'nớ', 'nức nở', 'nữa', 'nữa khi', 'nữa là', 'nữa rồi', 'oai oái', 'oái', 'pho', 'phè', 'phè phè', 'phía', 'phía bên', 'phía bạn', 'phía dưới', 'phía sau', 'phía trong', 'phía trên', 'phía trước', 'phóc', 'phót', 'phù hợp', 'phăn phắt', 'phương chi', 'phải', 'phải biết', 'phải chi', 'phải chăng', 'phải cách', 'phải cái', 'phải giờ', 'phải khi', 'phải không', 'phải lại', 'phải lời', 'phải người', 'phải như', 'phải rồi', 'phải tay', 'phần', 'phần lớn', 'phần nhiều', 'phần nào', 'phần sau', 'phần việc', 'phắt', 'phỉ phui', 'phỏng', 'phỏng như', 'phỏng nước', 'phỏng theo', 'phỏng tính', 'phốc', 'phụt', 'phứt', 'qua', 'qua chuyện', 'qua khỏi', 'qua lại', 'qua lần', 'qua ngày', 'qua tay', 'qua thì', 'qua đi', 'quan trọng', 'quan trọng vấn đề', 'quan tâm', 'quay', 'quay bước', 'quay lại', 'quay số', 'quay đi', 'quá', 'quá bán', 'quá bộ', 'quá giờ', 'quá lời', 'quá mức', 'quá nhiều', 'quá tay', 'quá thì', 'quá tin', 'quá trình', 'quá tuổi', 'quá đáng', 'quá ư', 'quả', 'quả là', 'quả thật', 'quả thế', 'quả vậy', 'quận', 'ra', 'ra bài', 'ra bộ', 'ra chơi', 'ra gì', 'ra lại', 'ra lời', 'ra ngôi', 'ra người', 'ra sao', 'ra tay', 'ra vào', 'ra ý', 'ra điều', 'ra đây', 'ren rén', 'riu ríu', 'riêng', 'riêng từng', 'riệt', 'rày', 'ráo', 'ráo cả', 'ráo nước', 'ráo trọi', 'rén', 'rén bước', 'rích', 'rón rén', 'rõ', 'rõ là', 'rõ thật', 'rút cục', 'răng', 'răng răng', 'rất', 'rất lâu', 'rằng', 'rằng là', 'rốt cuộc', 'rốt cục', 'rồi', 'rồi nữa', 'rồi ra', 'rồi sao', 'rồi sau', 'rồi tay', 'rồi thì', 'rồi xem', 'rồi đây', 'rứa', 'sa sả', 'sang', 'sang năm', 'sang sáng', 'sang tay', 'sao', 'sao bản', 'sao bằng', 'sao cho', 'sao vậy', 'sao đang', 'sau', 'sau chót', 'sau cuối', 'sau cùng', 'sau hết', 'sau này', 'sau nữa', 'sau sau', 'sau đây', 'sau đó', 'so', 'so với', 'song le', 'suýt', 'suýt nữa', 'sáng', 'sáng ngày', 'sáng rõ', 'sáng thế', 'sáng ý', 'sì', 'sì sì', 'sất', 'sắp', 'sắp đặt', 'sẽ', 'sẽ biết', 'sẽ hay', 'số', 'số cho biết', 'số cụ thể', 'số loại', 'số là', 'số người', 'số phần', 'số thiếu', 'sốt sột', 'sớm', 'sớm ngày', 'sở dĩ', 'sử dụng', 'sự', 'sự thế', 'sự việc', 'tanh', 'tanh tanh', 'tay', 'tay quay', 'tha hồ', 'tha hồ chơi', 'tha hồ ăn', 'than ôi', 'thanh', 'thanh ba', 'thanh chuyển', 'thanh không', 'thanh thanh', 'thanh tính', 'thanh điều kiện', 'thanh điểm', 'thay đổi', 'thay đổi tình trạng', 'theo', 'theo bước', 'theo như', 'theo tin', 'thi thoảng', 'thiếu', 'thiếu gì', 'thiếu điểm', 'thoạt', 'thoạt nghe', 'thoạt nhiên', 'thoắt', 'thuần', 'thuần ái', 'thuộc', 'thuộc bài', 'thuộc cách', 'thuộc lại', 'thuộc từ', 'thà', 'thà là', 'thà rằng', 'thành ra', 'thành thử', 'thái quá', 'tháng', 'tháng ngày', 'tháng năm', 'tháng tháng', 'thêm', 'thêm chuyện', 'thêm giờ', 'thêm vào', 'thì', 'thì giờ', 'thì là', 'thì phải', 'thì ra', 'thì thôi', 'thình lình', 'thích', 'thích cứ', 'thích thuộc', 'thích tự', 'thích ý', 'thím', 'thôi', 'thôi việc', 'thúng thắng', 'thương ôi', 'thường', 'thường bị', 'thường hay', 'thường khi', 'thường số', 'thường sự', 'thường thôi', 'thường thường', 'thường tính', 'thường tại', 'thường xuất hiện', 'thường đến', 'thảo hèn', 'thảo nào', 'thấp', 'thấp cơ', 'thấp thỏm', 'thấp xuống', 'thấy', 'thấy tháng', 'thẩy', 'thậm', 'thậm chí', 'thậm cấp', 'thậm từ', 'thật', 'thật chắc', 'thật là', 'thật lực', 'thật quả', 'thật ra', 'thật sự', 'thật thà', 'thật tốt', 'thật vậy', 'thế', 'thế chuẩn bị', 'thế là', 'thế lại', 'thế mà', 'thế nào', 'thế nên', 'thế ra', 'thế sự', 'thế thì', 'thế thôi', 'thế thường', 'thế thế', 'thế à', 'thế đó', 'thếch', 'thỉnh thoảng', 'thỏm', 'thốc', 'thốc tháo', 'thốt', 'thốt nhiên', 'thốt nói', 'thốt thôi', 'thộc', 'thời gian', 'thời gian sử dụng', 'thời gian tính', 'thời điểm', 'thục mạng', 'thứ', 'thứ bản', 'thứ đến', 'thửa', 'thực hiện', 'thực hiện đúng', 'thực ra', 'thực sự', 'thực tế', 'thực vậy', 'tin', 'tin thêm', 'tin vào', 'tiếp theo', 'tiếp tục', 'tiếp đó', 'tiện thể', 'toà', 'toé khói', 'toẹt', 'trong', 'trong khi', 'trong lúc', 'trong mình', 'trong ngoài', 'trong này', 'trong số', 'trong vùng', 'trong đó', 'trong ấy', 'tránh', 'tránh khỏi', 'tránh ra', 'tránh tình trạng', 'tránh xa', 'trên', 'trên bộ', 'trên dưới', 'trước', 'trước hết', 'trước khi', 'trước kia', 'trước nay', 'trước ngày', 'trước nhất', 'trước sau', 'trước tiên', 'trước tuổi', 'trước đây', 'trước đó', 'trả', 'trả của', 'trả lại', 'trả ngay', 'trả trước', 'trếu tráo', 'trển', 'trệt', 'trệu trạo', 'trỏng', 'trời đất ơi', 'trở thành', 'trừ phi', 'trực tiếp', 'trực tiếp làm', 'tuy', 'tuy có', 'tuy là', 'tuy nhiên', 'tuy rằng', 'tuy thế', 'tuy vậy', 'tuy đã', 'tuyệt nhiên', 'tuần tự', 'tuốt luốt', 'tuốt tuồn tuột', 'tuốt tuột', 'tuổi', 'tuổi cả', 'tuổi tôi', 'tà tà', 'tên', 'tên chính', 'tên cái', 'tên họ', 'tên tự', 'tênh', 'tênh tênh', 'tìm', 'tìm bạn', 'tìm cách', 'tìm hiểu', 'tìm ra', 'tìm việc', 'tình trạng', 'tính', 'tính cách', 'tính căn', 'tính người', 'tính phỏng', 'tính từ', 'tít mù', 'tò te', 'tôi', 'tôi con', 'tông tốc', 'tù tì', 'tăm tắp', 'tăng', 'tăng chúng', 'tăng cấp', 'tăng giảm', 'tăng thêm', 'tăng thế', 'tại', 'tại lòng', 'tại nơi', 'tại sao', 'tại tôi', 'tại vì', 'tại đâu', 'tại đây', 'tại đó', 'tạo', 'tạo cơ hội', 'tạo nên', 'tạo ra', 'tạo ý', 'tạo điều kiện', 'tấm', 'tấm bản', 'tấm các', 'tấn', 'tấn tới', 'tất cả', 'tất cả bao nhiêu', 'tất thảy', 'tất tần tật', 'tất tật', 'tập trung', 'tắp', 'tắp lự', 'tắp tắp', 'tọt', 'tỏ ra', 'tỏ vẻ', 'tốc tả', 'tối ư', 'tốt', 'tốt bạn', 'tốt bộ', 'tốt hơn', 'tốt mối', 'tốt ngày', 'tột', 'tột cùng', 'tớ', 'tới', 'tới gần', 'tới mức', 'tới nơi', 'tới thì', 'tức thì', 'tức tốc', 'từ', 'từ căn', 'từ giờ', 'từ khi', 'từ loại', 'từ nay', 'từ thế', 'từ tính', 'từ tại', 'từ từ', 'từ ái', 'từ điều', 'từ đó', 'từ ấy', 'từng', 'từng cái', 'từng giờ', 'từng nhà', 'từng phần', 'từng thời gian', 'từng đơn vị', 'từng ấy', 'tự', 'tự cao', 'tự khi', 'tự lượng', 'tự tính', 'tự tạo', 'tự vì', 'tự ý', 'tự ăn', 'tựu trung', 'veo', 'veo veo', 'việc', 'việc gì', 'vung thiên địa', 'vung tàn tán', 'vung tán tàn', 'và', 'vài', 'vài ba', 'vài người', 'vài nhà', 'vài nơi', 'vài tên', 'vài điều', 'vào', 'vào gặp', 'vào khoảng', 'vào lúc', 'vào vùng', 'vào đến', 'vâng', 'vâng chịu', 'vâng dạ', 'vâng vâng', 'vâng ý', 'vèo', 'vèo vèo', 'vì', 'vì chưng', 'vì rằng', 'vì sao', 'vì thế', 'vì vậy', 'ví bằng', 'ví dù', 'ví phỏng', 'ví thử', 'vô hình trung', 'vô kể', 'vô luận', 'vô vàn', 'vùng', 'vùng lên', 'vùng nước', 'văng tê', 'vượt', 'vượt khỏi', 'vượt quá', 'vạn nhất', 'vả chăng', 'vả lại', 'vấn đề', 'vấn đề quan trọng', 'vẫn', 'vẫn thế', 'vậy', 'vậy là', 'vậy mà', 'vậy nên', 'vậy ra', 'vậy thì', 'vậy ư', 'về', 'về không', 'về nước', 'về phần', 'về sau', 'về tay', 'vị trí', 'vị tất', 'vốn dĩ', 'với', 'với lại', 'với nhau', 'vở', 'vụt', 'vừa', 'vừa khi', 'vừa lúc', 'vừa mới', 'vừa qua', 'vừa rồi', 'vừa vừa', 'xa', 'xa cách', 'xa gần', 'xa nhà', 'xa tanh', 'xa tắp', 'xa xa', 'xa xả', 'xem', 'xem lại', 'xem ra', 'xem số', 'xin', 'xin gặp', 'xin vâng', 'xiết bao', 'xon xón', 'xoành xoạch', 'xoét', 'xoẳn', 'xoẹt', 'xuất hiện', 'xuất kì bất ý', 'xuất kỳ bất ý', 'xuể', 'xuống', 'xăm xúi', 'xăm xăm', 'xăm xắm', 'xảy ra', 'xềnh xệch', 'xệp', 'xử lý', 'yêu cầu', 'à', 'à này', 'à ơi', 'ào', 'ào vào', 'ào ào', 'á', 'á à', 'ái', 'ái chà', 'ái dà', 'áng', 'áng như', 'âu là', 'ít', 'ít biết', 'ít có', 'ít hơn', 'ít khi', 'ít lâu', 'ít nhiều', 'ít nhất', 'ít nữa', 'ít quá', 'ít ra', 'ít thôi', 'ít thấy', 'ô hay', 'ô hô', 'ô kê', 'ô kìa', 'ôi chao', 'ôi thôi', 'ông', 'ông nhỏ', 'ông tạo', 'ông từ', 'ông ấy', 'ông ổng', 'úi', 'úi chà', 'úi dào', 'ý', 'ý chừng', 'ý da', 'ý hoặc', 'ăn', 'ăn chung', 'ăn chắc', 'ăn chịu', 'ăn cuộc', 'ăn hết', 'ăn hỏi', 'ăn làm', 'ăn người', 'ăn ngồi', 'ăn quá', 'ăn riêng', 'ăn sáng', 'ăn tay', 'ăn trên', 'ăn về', 'đang', 'đang tay', 'đang thì', 'điều', 'điều gì', 'điều kiện', 'điểm', 'điểm chính', 'điểm gặp', 'điểm đầu tiên', 'đành đạch', 'đáng', 'đáng kể', 'đáng lí', 'đáng lý', 'đáng lẽ', 'đáng số', 'đánh giá', 'đánh đùng', 'đáo để', 'đâu', 'đâu có', 'đâu cũng', 'đâu như', 'đâu nào', 'đâu phải', 'đâu đâu', 'đâu đây', 'đâu đó', 'đây', 'đây này', 'đây rồi', 'đây đó', 'đã', 'đã hay', 'đã không', 'đã là', 'đã lâu', 'đã thế', 'đã vậy', 'đã đủ', 'đó', 'đó đây', 'đúng', 'đúng ngày', 'đúng ra', 'đúng tuổi', 'đúng với', 'đơn vị', 'đưa', 'đưa cho', 'đưa chuyện', 'đưa em', 'đưa ra', 'đưa tay', 'đưa tin', 'đưa tới', 'đưa vào', 'đưa về', 'đưa xuống', 'đưa đến', 'được', 'được cái', 'được lời', 'được nước', 'được tin', 'đại loại', 'đại nhân', 'đại phàm', 'đại để', 'đạt', 'đảm bảo', 'đầu tiên', 'đầy', 'đầy năm', 'đầy phè', 'đầy tuổi', 'đặc biệt', 'đặt', 'đặt làm', 'đặt mình', 'đặt mức', 'đặt ra', 'đặt trước', 'đặt để', 'đến', 'đến bao giờ', 'đến cùng', 'đến cùng cực', 'đến cả', 'đến giờ', 'đến gần', 'đến hay', 'đến khi', 'đến lúc', 'đến lời', 'đến nay', 'đến ngày', 'đến nơi', 'đến nỗi', 'đến thì', 'đến thế', 'đến tuổi', 'đến xem', 'đến điều', 'đến đâu', 'đều', 'đều bước', 'đều nhau', 'đều đều', 'để', 'để cho', 'để giống', 'để không', 'để lòng', 'để lại', 'để mà', 'để phần', 'để được', 'để đến nỗi', 'đối với', 'đồng thời', 'đủ', 'đủ dùng', 'đủ nơi', 'đủ số', 'đủ điều', 'đủ điểm', 'ơ', 'ơ hay', 'ơ kìa', 'ơi', 'ơi là', 'ư', 'ạ', 'ạ ơi', 'ấy', 'ấy là', 'ầu ơ', 'ắt', 'ắt hẳn', 'ắt là', 'ắt phải', 'ắt thật', 'ối dào', 'ối giời', 'ối giời ơi', 'ồ', 'ồ ồ', 'ổng', 'ớ', 'ớ này', 'ờ', 'ờ ờ', 'ở', 'ở lại', 'ở như', 'ở nhờ', 'ở năm', 'ở trên', 'ở vào', 'ở đây', 'ở đó', 'ở được', 'ủa', 'ứ hự', 'ứ ừ', 'ừ', 'ừ nhé', 'ừ thì', 'ừ ào', 'ừ ừ', 'ử']

In [114]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
from bertopic import BERTopic
from underthesea import sent_tokenize
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
Topics=['thể thao','giải trí','văn hóa','xã hội','chính trị','kinh tế','đời sống','công nghệ']
class TopicModel:
    def __init__(self,model_path):
        self.represent_model = SentenceTransformer(model_path)
        print('Load topic model successfully')

    def get_topic(self,paragraph):
     try:
        sentences = sent_tokenize(paragraph)
        segmented_sentences = [ViTokenizer.tokenize(sentence) for sentence in sentences]
        n = len(sentences)
        n_neighbors = max(2, min(15, n // 2))
        min_cluster_size = max(2, n // 5)
        vectorize_model=CountVectorizer(
            stop_words=vn_stopwords
        )
        representation_model=KeyBERTInspired()
        umap_model = UMAP(
            n_neighbors=n_neighbors,
            n_components=min(2, n - 1),
            min_dist=0.0,
            metric='cosine',
            random_state=42
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=1,
            metric='euclidean',
            prediction_data=True
        )
        topic_model = BERTopic(
            embedding_model=self.represent_model,
            representation_model=representation_model,
            vectorizer_model=vectorize_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            verbose=True,
            language='multilingual'
        )
        topics, prob = topic_model.fit_transform(segmented_sentences)
        freq = topic_model.get_topic_info()
        topic = freq.iloc[0]
        text =' '.join(topic['Representation'])
        topic_tokenize=[ViTokenizer.tokenize(topic) for topic in Topics]
        topic_embeddings= self.represent_model.encode(topic_tokenize)
        keyword_embedding=self.represent_model.encode([text])
        sims=cosine_similarity(
            keyword_embedding,topic_embeddings
        )
        top_k=3
        top_indices = np.argsort(sims)[::-1][0][:top_k]
        result="Chủ để của video trên thuộc lĩnh vực: "+', '.join(Topics[i] for i in top_indices)
        return result
     except:
          return "Không xác định được chủ để cho văn bản trên"


In [105]:
# topic_model=TopicModel('dangvantuan/vietnamese-embedding')
# p='''
#     tại nhật bản thì tài sản do người cao tuổi bị mất trí nhớ nắm giữ có thể vượt quá năm trăm nghìn tỷ yên tương đương với ba nghìn ba trăm tỷ đô la mỹ và năm hai nghìn không trăm ba mươi tài sản đó thì có thể là những bất động sản cổ phiếu trái phiếu tiền trong tài khoản ngân hàng có nguy cơ bị đóng băng tác động nhiều chiều đến thị trường phản ánh của phóng viên quang hưng thường trú đài truyền hình việt nam tại nhật bản. năm hai nghìn không trăm hai mươi tổng tài sản do người mắc chứng mất trí nhớ nắm giữ đã là một trăm tám mươi bảy nghìn tỷ yên đến năm hai nghìn không trăm ba mươi con số này dự kiến là hai trăm bốn mươi hai nghìn tỷ yên và nếu tính cả người cao tuổi bị suy giảm trí nhớ ở giai đoạn đầu thì tổng tài sản có thể lên tới năm trăm ba mươi ba nghìn tỷ yên tương đương với ba nghìn năm trăm năm mươi tỷ đô la mỹ nếu ngân hàng hoặc tổ chức tài chính xác định một người thiếu năng lực đưa ra quyết định tài khoản và giao dịch của họ có thể sẽ bị đọc băng. xử lý tài sản liên quan đến bất động sản cổ phiếu trái phiếu sẽ rất phức tạp và cần phải chỉ định người giám hộ hợp pháp nhiều trường hợp khó khăn trong việc chỉ định người giám hộ việc đóng băng này cũng ảnh hưởng đến thị trường tiêu dùng các khoản chi trả y tế viện nghiên cứu nhật bản ước tính mức tiêu dùng của người cao tuổi mắc chứng mất trí nhớ năm hai nghìn không trăm hai lăm đạt mười bốn phẩy bảy nghìn tỷ yên tức là khoảng chín mươi tám tỷ đô la mỹ theo sách chẳng của chính phủ nhật bản năm hai nghìn không trăm hai lăm số người suy giảm trí nhớ đạt năm phẩy hai triệu người. và sẽ tăng mười một phần trăm vào năm hai nghìn không trăm ba mươi nhật bản đang sửa đổi luật dân sự để đơn giản hoá hoạt động giám hộ tự nguyện và lập các quỹ tín thác để quản lý số tài sản khổng lồ này tài sản của người mất trí nhớ hoặc suy giảm nhận thức nghiêm trọng thì việc xử lý sẽ rất mất thời gian nhất là khi họ muốn chuyên đổi thành tiền để chi trả cho các dịch vụ y tế chữa bệnh tình trạng này có tác động trên một quy mô rộng hơn đó là nền kinh tế của người cao tuổi hoàng hưng phóng viên thường trú đài trình việt nam. đưa tin từ tokyo nhật bản.
#     '''
# final_topic=topic_model.get_topic(p)
# print(final_topic)

In [115]:
import os.path
import shutil
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import  CORSMiddleware
import subprocess
import nest_asyncio
from pyngrok import ngrok
import threading
from pytube import YouTube
from pydantic import BaseModel
import yt_dlp
origins = [
    "http://localhost:3000",
]
nest_asyncio.apply()
app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
class UploadItem(BaseModel):
  url:str
audio_to_text_model=Mp4_to_text('vinai/PhoWhisper-small')
summary_model=Summarizer('csebuetnlp/mT5_multilingual_XLSum')
topic_model=TopicModel('dangvantuan/vietnamese-embedding')
@app.post("/api/summary-file")
async def upload_file(file: UploadFile = File(...)):
    if os.path.exists('/content/audio/output3.mp3'):
        os.remove('/content/audio/output3.mp3')
    if os.path.exists('/content/audio_chunk/filemp3_chunk/output3'):
        shutil.rmtree('/content/audio_chunk/filemp3_chunk/output3')
    file_location = f"/content/audio/{file.filename}"
    with open(file_location, "wb") as f:
        f.write(await file.read())
    print('Processing audio to text')
    text=audio_to_text_model.op_vi(file_location)
    print("Processing topic modeling")
    topic=topic_model.get_topic(paragraph=text)
    print("Processing text summary")
    summary=summary_model.run(text)
    first_sent=summary.split('.')[0]
    return {'topic':first_sent+'\n'+topic,
            'summary':summary}
@app.post("/api/summary-link")
async def upload_file(item:UploadItem):
    url=item.url
    file_location = f"/content/audio/yt.mp4"
    if os.path.exists(file_location):
        os.remove(file_location)
    ydl_opts = {
    'format': 'best[ext=mp4]',
    'outtmpl': file_location
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
      ydl.download([url])
    if os.path.exists('/content/audio/output3.mp3'):
        os.remove('/content/audio/output3.mp3')
    if os.path.exists('/content/audio_chunk/filemp3_chunk/output3'):
        shutil.rmtree('/content/audio_chunk/filemp3_chunk/output3')
    text=audio_to_text_model.op_vi(file_location)
    topic=topic_model.get_topic(paragraph=text)
    summary=summary_model.run(text)
    first_sent=summary.split('.')[0]
    return {'topic':topic,
            'summary':summary}



Using device: cuda


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-45' coro=<start_server() done, defined at /tmp/ipykernel_703/3067220132.py:10> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_703/3067220132.py", line 15, in <cell line: 0>
    asyncio.get_event_loop().run_until_complete(start_server())
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 92, in run_until_complete
    self._run_once()
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 133, in _run_once
    handle._run()
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 396, in __wakeup
    self.__step()
  File "/usr/lib/python3.12/asyncio/tasks.py", lin

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


load model successfully
Using device: cuda


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Load summary model successfully !!!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Load topic model successfully


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio
ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)
nest_asyncio.apply()
async def start_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

asyncio.get_event_loop().run_until_complete(start_server())

Public URL: https://fifth-scanner-riptide.ngrok-free.dev


INFO:     Started server process [703]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Processing audio to text


Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-05-11 17:04:21,865 - BERTopic - Embedding - Transforming documents to embeddings.


Processing topic modeling


...

[youtube] zEylvkSQUNw: Downloading android vr player API JSON
[info] zEylvkSQUNw: Downloading 1 format(s): 18
[download] Destination: /content/audio/yt.mp4
[download] 100% of   11.49MiB in 00:00:00 at 16.66MiB/s  


Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-05-11 17:05:11,180 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-11 17:05:11,255 - BERTopic - Embedding - Completed ✓
2026-05-11 17:05:11,256 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-11 17:05:11,271 - BERTopic - Dimensionality - Completed ✓
2026-05-11 17:05:11,272 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-11 17:05:11,277 - BERTopic - Cluster - Completed ✓
2026-05-11 17:05:11,279 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-11 17:05:11,394 - BERTopic - Representation - Completed ✓


Số đoạn: 7
Đang xử lý đoạn 1/7...
Đang xử lý đoạn 2/7...
Đang xử lý đoạn 3/7...
Đang xử lý đoạn 4/7...
Đang xử lý đoạn 5/7...
Đang xử lý đoạn 6/7...
Đang xử lý đoạn 7/7...
INFO:     2402:800:6117:fe52:1567:67e0:30aa:719b:0 - "POST /api/summary-link HTTP/1.1" 200 OK
